# Summarize-then-Translate Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local Summarize-then-Translate baseline for English-to-Chinese cross-lingual dialogue summarization.

The ST pipeline uses two local small language model agents. Agent 1 reads the original English dialogue and generates a concise English summary. Agent 2 then translates the English summary into Chinese. The final output is a concise Chinese summary.

```text
English Dialogue
→ Agent 1: English Summarization Agent
→ Agent 2: Chinese Translation Agent
→ Final Chinese Summary
```

The pipeline consists of two agents:

```text
Agent 1: English Summarization Agent
Input: original English dialogue
Output: concise English summary

Agent 2: Chinese Translation Agent
Input: English summary from Agent 1
Output: final Chinese summary
```
This setup is used as a Summarize-then-Translate baseline. Unlike the Direct pipeline, ST explicitly creates an intermediate English summary before producing the Chinese summary. This allows us to inspect whether errors come from the summarization stage or the translation stage.

The local small language model is served through Ollama. The notebook controls the prompt design, agent workflow, input/output processing, intermediate output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the local model used for the Summarize-then-Translate baseline.

### Recommended model setup

This notebook uses one local small language model through Ollama for both agents:

- Agent 1: English Summarization Agent
- Agent 2: Chinese Translation Agent

```bash
ollama pull qwen3.5:27b
```

If qwen3.5:27b is too slow on your machine, you can use a smaller model for testing:

```bash
ollama pull "qwen3.5:9b"
```

Make sure the model names in the notebook match the models installed in Ollama:

```bash
SUMMARIZATION_MODEL = "qwen3.5:27b"
TRANSLATION_MODEL = "qwen3.5:27b"
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm


In [2]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# ST models
SUMMARIZATION_MODEL = "gemma3:27b"
TRANSLATION_MODEL = "gemma3:27b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# window project path
PROJECT_ROOT = Path(
    r"C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu"
)

# Gold set path
GOLD_SET_PATH = PROJECT_ROOT / "gold_set_50_zh_XSAMSum_bart.json"

# Output directory
OUTPUT_DIR = PROJECT_ROOT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files for ST baseline
FULL_OUTPUT_PATH = OUTPUT_DIR / "st_gemma27b_50samples.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "st_gemma27b_50samples.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "st_gemma27b_50samples_errors.jsonl"

print("Gold set path:", GOLD_SET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Gold set path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\gold_set_50_zh_XSAMSum_bart.json
Output directory: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu
Full JSONL output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\st_gemma27b_50samples.jsonl
Final CSV output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\st_gemma27b_50samples.csv
Error output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\st_gemma27b_50samples_errors.jsonl


In [3]:
print(GOLD_SET_PATH.exists())

True


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['gemma3:27b', 'qwen3.5:27b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: ST prompt templates

ENGLISH_SUMMARIZATION_PROMPT = """You are an English dialogue summarization agent.

Your task is to read the following English dialogue and generate a concise English summary.

Requirements:
- Summarize the main information in the dialogue.
- Write the summary in English.
- Keep the summary concise and faithful to the dialogue.
- Do not translate into Chinese.
- Do not add information that is not stated or clearly implied.
- Do not explain your reasoning.
- Output only the English summary.

Conciseness: 
- For simple dialogues, write one short English sentence.
- For complex dialogues, write at most two short English sentences.

English dialogue:
{dialogue}

English summary:
"""


CHINESE_TRANSLATION_PROMPT = """You are a Chinese translation agent.

Your task is to translate the English summary into Chinese.

Requirements:
- Translate the English summary into natural Chinese.
- Preserve the meaning of the English summary.
- Do not add new information.
- Do not remove important information.
- Do not explain your reasoning.
- Output only the final Chinese summary.
- Translate all English proper nouns, including speaker names, into standard Chinese transliteration.

STRICT TRANSLATION RULE:
- You MUST translate ALL English proper nouns and speaker names into standard Chinese characters.
- ABSOLUTELY NO English letters or names should appear in the final Chinese summary.

English summary:
{english_summary}

Chinese summary:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: ST agent functions

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def english_summarization_agent(dialogue: str) -> str:
    """Agent 1: English dialogue -> English summary."""
    prompt = fill_prompt(
        ENGLISH_SUMMARIZATION_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=SUMMARIZATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()


def chinese_translation_agent(english_summary: str) -> str:
    """Agent 2: English summary -> Chinese summary."""
    prompt = fill_prompt(
        CHINESE_TRANSLATION_PROMPT,
        {
            "english_summary": english_summary,
        },
    )

    response = call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: ST pipeline

def run_st_pipeline(example: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run the Summarize-then-Translate pipeline."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: English dialogue -> English summary
    english_summary = english_summarization_agent(dialogue)

    if verbose:
        print("\n" + "=" * 80)
        print(f"Sample ID: {sample_id}")
        print("=== Agent 1 Output: English Summary ===")
        print(english_summary)
        print("=" * 80 + "\n")

    # Agent 2: English summary -> Chinese summary
    final_chinese_summary = chinese_translation_agent(english_summary)

    if verbose:
        print("=== Agent 2 Output: Chinese Summary ===")
        print(final_chinese_summary)
        print("=" * 80 + "\n")

    return {
        "id": sample_id,
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,

        # Intermediate output from Agent 1
        "english_summary": english_summary,

        # Final output from Agent 2
        "final_summary": final_chinese_summary,

        # References
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,

        # Metadata
        "pipeline": "summarize_then_translate",
        "summarization_model": SUMMARIZATION_MODEL,
        "translation_model": TRANSLATION_MODEL,
        "num_model_calls": 2,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect whether errors come from the English summarization stage or the Chinese translation stage.

In [10]:
# Cell 9: Load first 5 examples from the gold set

def load_examples_from_gold_set(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the first n examples from the gold-set JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Gold set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"gold_{i+1:05d}",
            "test_index": item.get("test_index", ""),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_gold_set(GOLD_SET_PATH, n=50)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 50 examples.
First example:
{'id': 'gold_00001', 'test_index': 23, 'dialogue': "Anne: You were right, he was lying to me :/\nIrene: Oh no, what happened?\nJane: who? that Mark guy?\nAnne: yeah, he told me he's 30, today I saw his passport - he's 40\nIrene: You sure it's so important?\nAnne: he lied to me Irene", 'reference_english_summary': 'Mark lied to Anne about his age. Mark is 40.', 'reference_chinese_summary': '马克向安妮隐瞒了自己的年龄。他40岁了。'}


In [11]:
# Cell 10: Run the ST pipeline for the first example

result = run_st_pipeline(test_data[4], verbose=True)
result


Sample ID: gold_00005
=== Agent 1 Output: English Summary ===
Joyce shared a link for a cheap ticket, and both Michael and Edson expressed excitement about it, with Edson immediately planning to book.

=== Agent 2 Output: Chinese Summary ===
乔伊斯分享了一个廉价机票的链接，迈克尔和埃德森都表达了兴奋之情，埃德森立刻开始计划预订。



{'id': 'gold_00005',
 'test_index': 66,
 'dialogue': "Joyce: Check this out!\r\nJoyce: <link>\r\nMichael: That's cheap!\r\nEdson: No way! I'm booking my ticket now!! ",
 'english_summary': 'Joyce shared a link for a cheap ticket, and both Michael and Edson expressed excitement about it, with Edson immediately planning to book.',
 'final_summary': '乔伊斯分享了一个廉价机票的链接，迈克尔和埃德森都表达了兴奋之情，埃德森立刻开始计划预订。',
 'reference_english_summary': 'Edson is booking his ticket now.',
 'reference_chinese_summary': '埃德森正在订票。',
 'pipeline': 'summarize_then_translate',
 'summarization_model': 'gemma3:27b',
 'translation_model': 'gemma3:27b',
 'num_model_calls': 2}

In [12]:
# Cell 11: Print ST pipeline result clearly

def print_st_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1 Intermediate Output: English Summary ===")
    print(result["english_summary"])
    print()

    print("=== Agent 2 Final Output: Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Summarization model:", result["summarization_model"])
    print("Translation model:", result["translation_model"])
    print("Model calls:", result["num_model_calls"])


print_st_result(result)

=== Original Dialogue ===
Joyce: Check this out!
Joyce: <link>
Michael: That's cheap!
Edson: No way! I'm booking my ticket now!! 

=== Agent 1 Intermediate Output: English Summary ===
Joyce shared a link for a cheap ticket, and both Michael and Edson expressed excitement about it, with Edson immediately planning to book.

=== Agent 2 Final Output: Chinese Summary ===
乔伊斯分享了一个廉价机票的链接，迈克尔和埃德森都表达了兴奋之情，埃德森立刻开始计划预订。

=== Reference English Summary ===
Edson is booking his ticket now.

=== Reference Chinese Summary ===
埃德森正在订票。

=== Metadata ===
Pipeline: summarize_then_translate
Summarization model: gemma3:27b
Translation model: gemma3:27b
Model calls: 2


## 4. Save Results

This saves the intermediate English summary and the final Chinese summary.

In [13]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\st_gemma27b_50samples.jsonl
CSV output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\st_gemma27b_50samples.csv
Error output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\st_gemma27b_50samples_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed ST result to the JSONL output file.

If the notebook stops, already processed examples remain saved.

In [14]:
# Cell 13: Batch inference with ST pipeline
# Time stamp: 1m 29.8s

MAX_EXAMPLES = 50
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running ST pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        print(f"Skipping already processed sample: {sample_id}")
        continue

    try:
        record = run_st_pipeline(ex, verbose=True)

        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running ST pipeline:   0%|          | 0/50 [00:00<?, ?it/s]


Sample ID: gold_00001
=== Agent 1 Output: English Summary ===
Anne discovered Mark lied about his age, telling her he was 30 when his passport revealed he is 40, which upset her. Irene acknowledged Anne's disappointment with the deception.

=== Agent 2 Output: Chinese Summary ===
安妮发现马克对自己的年龄说了谎，他告诉她自己30岁，但他的护照显示他实际年龄是40岁，这让她感到难过。艾琳承认安妮对这次欺骗感到失望。


Sample ID: gold_00002
=== Agent 1 Output: English Summary ===
Mary asked Carter for some money, and Carter said he could lend it to her in an hour as he is currently at the train station.

=== Agent 2 Output: Chinese Summary ===
玛丽向卡特请求了一些钱，卡特说他可以在一个小时后借给她，因为他现在在火车站。


Sample ID: gold_00003
=== Agent 1 Output: English Summary ===
Tina was delayed at the airport and had a talkative pilot on her flight home, while Ala is going to an important meeting and asked Tina to wish her luck. Tina agreed to hear how the meeting went.

=== Agent 2 Output: Chinese Summary ===
蒂娜在机场延误，并且在回家的航班上遇到了一位健谈的飞行员。艾拉将要参加一个重要的会议，并请蒂娜为她祝好运。蒂娜同意听她讲述会议的情况。


Sample ID

## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.

For the ST pipeline, the CSV also includes the intermediate English summary from Agent 1.

In [15]:
# Cell 14: Export ST summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "test_index": record.get("test_index", ""),
        "dialogue": record.get("dialogue", ""),

        # Intermediate output from Agent 1
        "english_summary": record.get("english_summary", ""),

        # Final output from Agent 2
        "final_summary": record.get("final_summary", ""),

        # References
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),

        # Metadata
        "pipeline": record.get("pipeline", "summarize_then_translate"),
        "summarization_model": record.get("summarization_model", ""),
        "translation_model": record.get("translation_model", ""),
        "num_model_calls": record.get("num_model_calls", 2),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\st_gemma27b_50samples.csv


,id,test_index,dialogue,english_summary,final_summary,reference_english_summary,reference_chinese_summary,pipeline,summarization_model,translation_model,num_model_calls
0,gold_00001,23,"Anne: You were right, he was lying to me :/\nI...","Anne discovered Mark lied about his age, telli...",安妮发现马克对自己的年龄说了谎，他告诉她自己30岁，但他的护照显示他实际年龄是40岁，这让她...,Mark lied to Anne about his age. Mark is 40.,马克向安妮隐瞒了自己的年龄。他40岁了。,summarize_then_translate,gemma3:27b,gemma3:27b,2
1,gold_00002,30,"Mary: hey, im kinda broke, lend me a few box\r...","Mary asked Carter for some money, and Carter s...",玛丽向卡特请求了一些钱，卡特说他可以在一个小时后借给她，因为他现在在火车站。,Mary ran out of money. Carter is going to lend...,玛丽的钱用完了，卡特打算一小时后借给她一点。,summarize_then_translate,gemma3:27b,gemma3:27b,2
2,gold_00003,39,"Tina: I'll tell you something, this Emirate st...",Tina was delayed at the airport and had a talk...,蒂娜在机场延误，并且在回家的航班上遇到了一位健谈的飞行员。艾拉将要参加一个重要的会议，并请蒂...,Tina will catch the evening flight back home. ...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。,summarize_then_translate,gemma3:27b,gemma3:27b,2
3,gold_00004,65,Ana: You sleeping?\r\nCatherine: Not yet.\r\nA...,Ana and Catherine plan to visit their grandma ...,安娜和凯瑟琳计划明天去看望她们的奶奶，凯瑟琳会在她醒来后给安娜打电话。然后她们互相道了晚安。,Ana wants to visit grandma tomorrow. Catherine...,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。,summarize_then_translate,gemma3:27b,gemma3:27b,2
4,gold_00005,66,Joyce: Check this out!\r\nJoyce: <link>\r\nMic...,"Joyce shared a link for a cheap ticket, prompt...",乔伊斯分享了一个廉价机票的链接，促使迈克尔承认价格，而埃德森立即预订了一张机票。,Edson is booking his ticket now.,埃德森正在订票。,summarize_then_translate,gemma3:27b,gemma3:27b,2
5,gold_00006,67,Jane: google maps says it is at least 3h <file...,"Jane and Steven are planning to meet, and Jane...",简和史蒂文计划见面，简希望因为路途遥远而将见面时间提前到下午4点30分，而不是5点，史蒂文同...,Jane wants to leave at 4.30 instead of 5 becau...,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...,summarize_then_translate,gemma3:27b,gemma3:27b,2
6,gold_00007,78,"Fiona: Are you free?\r\nTina: Yes, what's up?\...",Fiona wants to make Tina’s tart for dinner wit...,菲奥娜想为克里斯做晚餐，想做蒂娜的馅饼，于是向蒂娜寻求帮助，并承认她买好了派皮，之前因为过度...,Fiona wants to prepare dinner for Chris. She i...,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。,summarize_then_translate,gemma3:27b,gemma3:27b,2
7,gold_00008,86,Olafur: are we doing anything for New Year's E...,The group is planning to go to a classy New Ye...,该团队计划前往苏荷区的一家俱乐部参加一场高档的新年晚会，可能是在“蒂芙尼的早餐”俱乐部，他们...,"Nathalie, Olafur and Zoe are planning the New ...",娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...,summarize_then_translate,gemma3:27b,gemma3:27b,2
8,gold_00009,120,"John: wanna go see ""A Star is Born"" on Wed?\r\...","John and Joan agreed to see ""A Star is Born"" o...",约翰和琼同意在周四晚上8点左右观看电影《一个明星的诞生》，此前琼表示周三她很忙。约翰会把电影...,"Joan and John are going to watch ""A Star is Bo...",琼和约翰星期四晚上8点左右去看《一个明星的诞生》。,summarize_then_translate,gemma3:27b,gemma3:27b,2
9,gold_00010,137,Peyton: I have been asking you to bring that v...,Cameron is unable to bring Peyton a video game...,卡梅隆无法给佩顿送游戏机，因为他需要出城另待一周，这促使佩顿建议通过快递寄送。佩顿随后要求卡...,Peyton is expecting Cameron to bring the video...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。,summarize_then_translate,gemma3:27b,gemma3:27b,2


In [16]:
# Cell 15: Compare ST outputs with references

comparison_columns = [
    "id",
    "test_index",
    "english_summary",
    "reference_english_summary",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,test_index,english_summary,reference_english_summary,final_summary,reference_chinese_summary
0,gold_00001,23,"Anne discovered Mark lied about his age, telli...",Mark lied to Anne about his age. Mark is 40.,安妮发现马克对自己的年龄说了谎，他告诉她自己30岁，但他的护照显示他实际年龄是40岁，这让她...,马克向安妮隐瞒了自己的年龄。他40岁了。
1,gold_00002,30,"Mary asked Carter for some money, and Carter s...",Mary ran out of money. Carter is going to lend...,玛丽向卡特请求了一些钱，卡特说他可以在一个小时后借给她，因为他现在在火车站。,玛丽的钱用完了，卡特打算一小时后借给她一点。
2,gold_00003,39,Tina was delayed at the airport and had a talk...,Tina will catch the evening flight back home. ...,蒂娜在机场延误，并且在回家的航班上遇到了一位健谈的飞行员。艾拉将要参加一个重要的会议，并请蒂...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。
3,gold_00004,65,Ana and Catherine plan to visit their grandma ...,Ana wants to visit grandma tomorrow. Catherine...,安娜和凯瑟琳计划明天去看望她们的奶奶，凯瑟琳会在她醒来后给安娜打电话。然后她们互相道了晚安。,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。
4,gold_00005,66,"Joyce shared a link for a cheap ticket, prompt...",Edson is booking his ticket now.,乔伊斯分享了一个廉价机票的链接，促使迈克尔承认价格，而埃德森立即预订了一张机票。,埃德森正在订票。
5,gold_00006,67,"Jane and Steven are planning to meet, and Jane...",Jane wants to leave at 4.30 instead of 5 becau...,简和史蒂文计划见面，简希望因为路途遥远而将见面时间提前到下午4点30分，而不是5点，史蒂文同...,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...
6,gold_00007,78,Fiona wants to make Tina’s tart for dinner wit...,Fiona wants to prepare dinner for Chris. She i...,菲奥娜想为克里斯做晚餐，想做蒂娜的馅饼，于是向蒂娜寻求帮助，并承认她买好了派皮，之前因为过度...,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。
7,gold_00008,86,The group is planning to go to a classy New Ye...,"Nathalie, Olafur and Zoe are planning the New ...",该团队计划前往苏荷区的一家俱乐部参加一场高档的新年晚会，可能是在“蒂芙尼的早餐”俱乐部，他们...,娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...
8,gold_00009,120,"John and Joan agreed to see ""A Star is Born"" o...","Joan and John are going to watch ""A Star is Bo...",约翰和琼同意在周四晚上8点左右观看电影《一个明星的诞生》，此前琼表示周三她很忙。约翰会把电影...,琼和约翰星期四晚上8点左右去看《一个明星的诞生》。
9,gold_00010,137,Cameron is unable to bring Peyton a video game...,Peyton is expecting Cameron to bring the video...,卡梅隆无法给佩顿送游戏机，因为他需要出城另待一周，这促使佩顿建议通过快递寄送。佩顿随后要求卡...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。


In [17]:
# Cell 16: Inspect ST outputs

if not df.empty:
    inspection_columns = [
        "id",
        "test_index",
        "dialogue",
        "english_summary",
        "reference_english_summary",
        "final_summary",
        "reference_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,test_index,dialogue,english_summary,reference_english_summary,final_summary,reference_chinese_summary
0,gold_00001,23,"Anne: You were right, he was lying to me :/\nI...","Anne discovered Mark lied about his age, telli...",Mark lied to Anne about his age. Mark is 40.,安妮发现马克对自己的年龄说了谎，他告诉她自己30岁，但他的护照显示他实际年龄是40岁，这让她...,马克向安妮隐瞒了自己的年龄。他40岁了。
1,gold_00002,30,"Mary: hey, im kinda broke, lend me a few box\r...","Mary asked Carter for some money, and Carter s...",Mary ran out of money. Carter is going to lend...,玛丽向卡特请求了一些钱，卡特说他可以在一个小时后借给她，因为他现在在火车站。,玛丽的钱用完了，卡特打算一小时后借给她一点。
2,gold_00003,39,"Tina: I'll tell you something, this Emirate st...",Tina was delayed at the airport and had a talk...,Tina will catch the evening flight back home. ...,蒂娜在机场延误，并且在回家的航班上遇到了一位健谈的飞行员。艾拉将要参加一个重要的会议，并请蒂...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。
3,gold_00004,65,Ana: You sleeping?\r\nCatherine: Not yet.\r\nA...,Ana and Catherine plan to visit their grandma ...,Ana wants to visit grandma tomorrow. Catherine...,安娜和凯瑟琳计划明天去看望她们的奶奶，凯瑟琳会在她醒来后给安娜打电话。然后她们互相道了晚安。,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。
4,gold_00005,66,Joyce: Check this out!\r\nJoyce: <link>\r\nMic...,"Joyce shared a link for a cheap ticket, prompt...",Edson is booking his ticket now.,乔伊斯分享了一个廉价机票的链接，促使迈克尔承认价格，而埃德森立即预订了一张机票。,埃德森正在订票。
5,gold_00006,67,Jane: google maps says it is at least 3h <file...,"Jane and Steven are planning to meet, and Jane...",Jane wants to leave at 4.30 instead of 5 becau...,简和史蒂文计划见面，简希望因为路途遥远而将见面时间提前到下午4点30分，而不是5点，史蒂文同...,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...
6,gold_00007,78,"Fiona: Are you free?\r\nTina: Yes, what's up?\...",Fiona wants to make Tina’s tart for dinner wit...,Fiona wants to prepare dinner for Chris. She i...,菲奥娜想为克里斯做晚餐，想做蒂娜的馅饼，于是向蒂娜寻求帮助，并承认她买好了派皮，之前因为过度...,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。
7,gold_00008,86,Olafur: are we doing anything for New Year's E...,The group is planning to go to a classy New Ye...,"Nathalie, Olafur and Zoe are planning the New ...",该团队计划前往苏荷区的一家俱乐部参加一场高档的新年晚会，可能是在“蒂芙尼的早餐”俱乐部，他们...,娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...
8,gold_00009,120,"John: wanna go see ""A Star is Born"" on Wed?\r\...","John and Joan agreed to see ""A Star is Born"" o...","Joan and John are going to watch ""A Star is Bo...",约翰和琼同意在周四晚上8点左右观看电影《一个明星的诞生》，此前琼表示周三她很忙。约翰会把电影...,琼和约翰星期四晚上8点左右去看《一个明星的诞生》。
9,gold_00010,137,Peyton: I have been asking you to bring that v...,Cameron is unable to bring Peyton a video game...,Peyton is expecting Cameron to bring the video...,卡梅隆无法给佩顿送游戏机，因为他需要出城另待一周，这促使佩顿建议通过快递寄送。佩顿随后要求卡...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。
